In [ ]:
#pip install xgboost scikit-learn
import pandas as pd
import numpy as np

# 原始数据 + 全球大盘特征 (Global Box Office Market Size in $B)
# 数据参考：Statista/Gower Street 全球年度票房统计
data = {
    'title': ['速1', '速2', '速3', '速4', '速5', '速6', '速7', '速8', '速9', '速10'],
    'budget': [38, 76, 85, 85, 125, 160, 190, 250, 200, 340],
    'marketing': [20, 35, 40, 50, 80, 100, 150, 160, 140, 175],
    'cast_fans': [1.8, 1.1, 0.3, 2.8, 13.5, 21.0, 21.0, 52.0, 40.0, 45.0],
    'douban': [8.1, 7.5, 7.0, 7.8, 8.5, 7.9, 8.4, 7.0, 5.1, 6.1],
    'rotten_tomatoes': [54, 36, 38, 29, 78, 71, 81, 67, 59, 56],
    # 核心新增：全球年度总票房 (单位：$B)
    'market_size': [19.0, 20.2, 25.5, 29.4, 32.6, 35.9, 39.1, 40.5, 21.3, 33.9], 
    'gross': [207, 236, 158, 363, 626, 788, 1515, 1236, 726, 704]
}

df = pd.DataFrame(data)
print("数据集已成功引入外部大盘特征（Market Size）。")

数据集已成功引入外部大盘特征（Market Size）。


In [2]:
# 1. 滞后特征 (上一部票房)
df['prev_gross'] = df['gross'].shift(1).fillna(df['gross'].mean())

# 2. 题材转向 (0: 赛车, 1: 动作)
df['genre_shift'] = [0, 0, 0, 1, 1, 1, 1, 1, 1, 1]

# 3. 计算相关性（验证方案A是否有效）
correlation = df['market_size'].corr(df['gross'])
print(f"全球大盘与系列票房的相关系数为: {correlation:.2f}")

# 4. 确定最终特征列表
features = ['budget', 'marketing', 'cast_fans', 'douban', 'rotten_tomatoes', 'market_size', 'prev_gross', 'genre_shift']
X = df[features]
y = df['gross']

全球大盘与系列票房的相关系数为: 0.80


In [3]:
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error

loo = LeaveOneOut()
mae_lr, mae_xgb = [], []

for train_idx, test_idx in loo.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # 线性回归 (加入大盘后，线性回归通常能更好地处理宏观趋势)
    lr = LinearRegression().fit(X_train, y_train)
    mae_lr.append(abs(lr.predict(X_test)[0] - y_test.values[0]))
    
    # XGBoost
    xg = xgb.XGBRegressor(n_estimators=30, max_depth=3, learning_rate=0.1, random_state=42)
    xg.fit(X_train, y_train)
    mae_xgb.append(abs(xg.predict(X_test)[0] - y_test.values[0]))

print(f"优化后线性回归 MAE: ${np.mean(mae_lr):.2f}M")
print(f"优化后 XGBoost MAE: ${np.mean(mae_xgb):.2f}M")

优化后线性回归 MAE: $877.69M
优化后 XGBoost MAE: $186.15M


In [4]:
# 假设速11数据：成本300, 宣发160, 粉丝55, 豆瓣6.0, 烂番茄60, 大盘预期38.0, 前作704, 动作1
future_11 = pd.DataFrame([[300, 160, 55.0, 6.0, 60, 38.0, 704, 1]], columns=features)

# 使用全量数据重新训练
final_model = xgb.XGBRegressor(n_estimators=30, max_depth=3, learning_rate=0.1)
final_model.fit(X, y)

pred_11 = final_model.predict(future_11)
print(f"《速度与激情11》全球票房最终预测: ${pred_11[0]:.2f}M")




《速度与激情11》全球票房最终预测: $1053.77M


### 修正方法

In [15]:
import pandas as pd
import numpy as np

# 1-10部数据拆分（估算值，用于模型训练）
# 检查你的数据初始化部分，确保 'gross' 在里面
data = {
    'title': ['速1', '速2', '速3', '速4', '速5', '速6', '速7', '速8', '速9', '速10'],
    'douban': [8.1, 7.5, 7.0, 7.8, 8.5, 7.9, 8.4, 7.0, 5.1, 6.1],
    'china_marketing': [5, 8, 10, 15, 25, 35, 50, 60, 60, 70],
    'rt_score': [54, 36, 38, 29, 78, 71, 81, 67, 59, 56],
    'cast_fans': [1.8, 1.1, 0.3, 2.8, 13.5, 21.0, 21.0, 52.0, 40.0, 45.0],
    'top_grosser': [974, 1104, 1066, 2744, 1328, 1276, 2068, 1236, 1906, 1445],
    'china_gross': [0, 0, 0, 4.5, 38, 66, 390, 392, 215, 139],
    'foreign_gross': [207, 236, 158, 358, 588, 722, 1125, 844, 511, 565],
    'gross': [207, 236, 158, 363, 626, 788, 1515, 1236, 726, 704] # <--- 确保这一行存在！
}
df = pd.DataFrame(data)


In [16]:

from sklearn.linear_model import Ridge

# --- 模型 1：预测中国票房 (专注于口碑与本土宣发) ---
X_china = df[['douban', 'china_marketing']]
y_china = np.log1p(df['china_gross']) # 对数化处理

model_china = Ridge(alpha=1.0).fit(X_china, y_china)

# --- 模型 2：预测海外票房 (专注于全球IP影响力和市场天花板) ---
X_foreign = df[['rt_score', 'cast_fans', 'top_grosser']]
y_foreign = np.log1p(df['foreign_gross'])

model_foreign = Ridge(alpha=1.0).fit(X_foreign, y_foreign)

In [17]:
def predict_global(douban, c_market, rt, fans, top):
    # 将输入封装成 DataFrame，并指定与训练时完全一致的列名
    input_china = pd.DataFrame([[douban, c_market]], columns=['douban', 'china_marketing'])
    input_foreign = pd.DataFrame([[rt, fans, top]], columns=['rt_score', 'cast_fans', 'top_grosser'])
    
    # 预测并还原
    p_china = np.expm1(model_china.predict(input_china))[0]
    p_foreign = np.expm1(model_foreign.predict(input_foreign))[0]
    
    return p_china + p_foreign

# 批量计算 1-10 部的集成预测值
df['total_pred'] = df.apply(lambda x: predict_global(x['douban'], x['china_marketing'], 
                                                     x['rt_score'], x['cast_fans'], 
                                                     x['top_grosser']), axis=1)

final_mae = np.mean(abs(df['total_pred'] - df['gross']))
print(f"分市场建模后的全球总票房 MAE: ${final_mae:.2f}M")


分市场建模后的全球总票房 MAE: $117.93M


In [18]:
# 封装成带有列名的 DataFrame
input_11_china = pd.DataFrame([[6.0, 70]], columns=['douban', 'china_marketing'])
input_11_foreign = pd.DataFrame([[60, 55.0, 2000.0]], columns=['rt_score', 'cast_fans', 'top_grosser'])

# 使用 DataFrame 进行预测
pred_11_china = np.expm1(model_china.predict(input_11_china))[0]
pred_11_foreign = np.expm1(model_foreign.predict(input_11_foreign))[0]

print(f"《速11》中国区预测: ${pred_11_china:.2f}M")
print(f"《速11》海外区预测: ${pred_11_foreign:.2f}M")
print(f"《速11》全球合并预测: ${pred_11_china + pred_11_foreign:.2f}M")

《速11》中国区预测: $524.92M
《速11》海外区预测: $904.16M
《速11》全球合并预测: $1429.08M
